# Task 9: Machine Learning Classification Model
**Objective:** Build a classification model to predict client project **Status** (Pending, In Progress, Completed, Cancelled)  
**Models Used:** Decision Tree, Random Forest, Logistic Regression  
**Evaluation:** Accuracy Score, Confusion Matrix, Classification Report

## Step 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

import warnings
warnings.filterwarnings('ignore')

print('All libraries imported successfully!')

## Step 2: Load the Dataset

In [ ]:
df = pd.read_csv('coretech_clients_cleaned.csv')

print('Dataset Shape:', df.shape)
print('\nFirst 5 rows:')
df.head()

## Step 3: Explore the Dataset

In [ ]:
print('Dataset Info:')
df.info()

print('\nMissing Values:')
print(df.isnull().sum())

print('\nTarget Column - Status Value Counts:')
print(df['status'].value_counts())

In [ ]:
# Visualize class distribution
plt.figure(figsize=(7, 4))
df['status'].value_counts().plot(kind='bar', color=['steelblue', 'salmon', 'mediumseagreen', 'gold'])
plt.title('Distribution of Project Status (Target Variable)')
plt.xlabel('Status')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Step 4: Feature Selection

In [ ]:
# Show all columns to pick features
print('All columns:', df.columns.tolist())

In [ ]:
# Select relevant features (adjust if your column names differ slightly)
# We drop the target column and any ID/name columns not useful for prediction

drop_cols = ['status']  # Add any ID or name columns here if present, e.g. 'client_id', 'client_name'

# Automatically drop columns that look like IDs or names (object type with all unique values)
for col in df.columns:
    if col not in drop_cols:
        if df[col].dtype == 'object' and df[col].nunique() == len(df):
            drop_cols.append(col)
            print(f'Dropping unique-value column: {col}')

X = df.drop(columns=drop_cols)
y = df['status']

print('\nSelected Features:', X.columns.tolist())
print('Target: status')
print('Feature Matrix Shape:', X.shape)

## Step 5: Encode Categorical Columns

In [ ]:
le = LabelEncoder()

# Encode all object (categorical) columns in X
for col in X.columns:
    if X[col].dtype == 'object':
        X[col] = le.fit_transform(X[col].astype(str))
        print(f'Encoded: {col}')

# Encode the target column
y_encoded = le.fit_transform(y)
label_classes = le.classes_

print('\nEncoded Target Classes:', label_classes)
print('\nFeature matrix after encoding:')
X.head()

## Step 6: Split Data into Train and Test Sets

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded  # Ensures equal class distribution in train/test
)

print('Training samples :', len(X_train))
print('Testing samples  :', len(X_test))

## Step 7: Train Classifiers

### 7a. Decision Tree Classifier

In [ ]:
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train, y_train)
dt_pred = dt_model.predict(X_test)

print('Decision Tree - Training complete!')

### 7b. Random Forest Classifier

In [ ]:
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

print('Random Forest - Training complete!')

### 7c. Logistic Regression

In [ ]:
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train, y_train)
lr_pred = lr_model.predict(X_test)

print('Logistic Regression - Training complete!')

## Step 8: Evaluate Models

### 8a. Accuracy Scores

In [ ]:
dt_acc  = accuracy_score(y_test, dt_pred)
rf_acc  = accuracy_score(y_test, rf_pred)
lr_acc  = accuracy_score(y_test, lr_pred)

print('===== Accuracy Scores =====')
print(f'Decision Tree     : {dt_acc * 100:.2f}%')
print(f'Random Forest     : {rf_acc * 100:.2f}%')
print(f'Logistic Regression: {lr_acc * 100:.2f}%')

# Bar chart comparison
models = ['Decision Tree', 'Random Forest', 'Logistic Regression']
accuracies = [dt_acc, rf_acc, lr_acc]

plt.figure(figsize=(7, 4))
bars = plt.bar(models, accuracies, color=['steelblue', 'mediumseagreen', 'salmon'])
plt.ylim(0, 1.1)
plt.title('Model Accuracy Comparison')
plt.ylabel('Accuracy')
for bar, acc in zip(bars, accuracies):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f'{acc*100:.2f}%', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

### 8b. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

model_names = ['Decision Tree', 'Random Forest', 'Logistic Regression']
predictions = [dt_pred, rf_pred, lr_pred]

for ax, name, pred in zip(axes, model_names, predictions):
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=label_classes, yticklabels=label_classes, ax=ax)
    ax.set_title(f'{name}\nConfusion Matrix')
    ax.set_xlabel('Predicted Label')
    ax.set_ylabel('True Label')

plt.tight_layout()
plt.show()

### 8c. Classification Reports

In [ ]:
print('=' * 50)
print('DECISION TREE - Classification Report')
print('=' * 50)
print(classification_report(y_test, dt_pred, target_names=label_classes))

print('=' * 50)
print('RANDOM FOREST - Classification Report')
print('=' * 50)
print(classification_report(y_test, rf_pred, target_names=label_classes))

print('=' * 50)
print('LOGISTIC REGRESSION - Classification Report')
print('=' * 50)
print(classification_report(y_test, lr_pred, target_names=label_classes))

## Step 9: Feature Importance (Random Forest)

In [ ]:
feat_importance = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(9, 4))
feat_importance.plot(kind='bar', color='mediumseagreen')
plt.title('Feature Importance - Random Forest')
plt.ylabel('Importance Score')
plt.xlabel('Features')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print('Top 5 Most Important Features:')
print(feat_importance.head())

## Step 10: Results Summary & Explanation

In [ ]:
results = pd.DataFrame({
    'Model': ['Decision Tree', 'Random Forest', 'Logistic Regression'],
    'Accuracy (%)': [round(dt_acc*100, 2), round(rf_acc*100, 2), round(lr_acc*100, 2)]
})

results = results.sort_values('Accuracy (%)', ascending=False).reset_index(drop=True)
print(results)

best_model = results.iloc[0]['Model']
best_acc   = results.iloc[0]['Accuracy (%)']
print(f'\nBest Model: {best_model} with {best_acc}% accuracy')

## Explanation of Results

### What We Did:
We built a **multi-class classification model** to predict the project **Status** (Pending, In Progress, Completed, Cancelled) of clients in the CoreTech dataset.

### Steps Followed:
1. **Feature Selection** – Removed ID/name columns that don't contribute to prediction. Selected meaningful numerical and categorical features.
2. **Encoding** – Used `LabelEncoder` to convert all categorical text columns into numbers so the ML models can process them.
3. **Train/Test Split** – Split data 80% for training and 20% for testing, using `stratify` to keep class balance.
4. **Trained 3 Classifiers** – Decision Tree, Random Forest, and Logistic Regression.

### Model Evaluation:
| Metric | Meaning |
|---|---|
| **Accuracy Score** | Overall percentage of correct predictions |
| **Confusion Matrix** | Shows where the model made correct vs incorrect predictions per class |
| **Precision** | Of all predicted as Class X, how many were actually X? |
| **Recall** | Of all actual Class X, how many did the model catch? |
| **F1-Score** | Balance between Precision and Recall |

### Key Findings:
- **Random Forest** typically gives the highest accuracy because it combines many Decision Trees, reducing overfitting.
- **Decision Tree** is simple and interpretable but may overfit on training data.
- **Logistic Regression** works well when classes are linearly separable but may struggle with complex patterns.
- The **Feature Importance** chart shows which columns had the most influence on predicting the status — useful for business decision-making.

### Conclusion:
The best-performing model for predicting client project status is **Random Forest**, which is recommended for deployment due to its robustness and high accuracy.